Do yesterday's QQQ returns help predict today's QQQ returns?

In [1]:
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

print("Working environment ready.")

Working environment ready.


In [2]:
ticker = "QQQ"

qqq = yf.download(
    ticker,
    start="2015-01-01",
    auto_adjust=True,
    progress=False
)

prices = qqq[["Close"]].copy()
prices = prices.rename(columns={"Close": "QQQ_close"})

prices["QQQ_log_return"] = np.log(
    prices["QQQ_close"] / prices["QQQ_close"].shift(1)
)

returns = prices["QQQ_log_return"].dropna()

returns.head()

Date
2015-01-05   -0.014777
2015-01-06   -0.013499
2015-01-07    0.012809
2015-01-08    0.018959
2015-01-09   -0.006605
Name: QQQ_log_return, dtype: float64

Import relevant machine-learning libraries.

In [3]:
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error

Create lagged returns.

In [4]:
ar_data = pd.DataFrame({
    "return_t": returns,
    "return_lag_1": returns.shift(1)
}).dropna()

ar_data.head()

,return_t,return_lag_1
Date,,
2015-01-06,-0.013499,-0.014777
2015-01-07,0.012809,-0.013499
2015-01-08,0.018959,0.012809
2015-01-09,-0.006605,0.018959
2015-01-12,-0.010482,-0.006605


Split into train and test sets.

In [5]:
split_index = int(len(ar_data) * 0.8)

train = ar_data.iloc[:split_index]
test = ar_data.iloc[split_index:]

print(f"Training set size: {len(train)}")
print(f"Test set size: {len(test)}")

print(f"Training set date range: {train.index.min().date()} to {train.index.max().date()}")
print(f"Test set date range: {test.index.min().date()} to {test.index.max().date()}")

Training set size: 2300
Test set size: 575
Training set date range: 2015-01-06 to 2024-02-26
Test set date range: 2024-02-27 to 2026-06-11


The AR(1) model is trained on the first 80% of the return data and evaluated on the final 20%. Since this is time-series
data, the split is chronological rather than random. This better reflects a realistic forecast setting where only past
information is available when predicting future returns.

An AR(1) model is: r_t = c + φ r_{t-1} + ε_t

In [6]:
y_train = train["return_t"]
X_train = train[["return_lag_1"]]

X_train = sm.add_constant(X_train, has_constant="add")

ar1_model = sm.OLS(y_train, X_train).fit()

print(ar1_model.summary())

                            OLS Regression Results                            
Dep. Variable:               return_t   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     29.04
Date:                Thu, 11 Jun 2026   Prob (F-statistic):           7.82e-08
Time:                        21:38:57   Log-Likelihood:                 6573.7
No. Observations:                2300   AIC:                        -1.314e+04
Df Residuals:                    2298   BIC:                        -1.313e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0007      0.000      2.550   

Extract key information.

In [7]:
c_hat = ar1_model.params["const"]
phi_hat = ar1_model.params["return_lag_1"]
p_value = ar1_model.pvalues["return_lag_1"]
r_squared = ar1_model.rsquared

print(f"Estimated constant (c): {c_hat:.8f}")
print(f"Estimated AR(1) coefficient (φ): {phi_hat:.8f}")
print(f"P-value for AR(1) coefficient: {p_value:.6f}")
print(f"Training R-squared: {r_squared:.6f}")

Estimated constant (c): 0.00073934
Estimated AR(1) coefficient (φ): -0.11167936
P-value for AR(1) coefficient: 0.000000
Training R-squared: 0.012479


The AR(1) coefficient measures whether yesterday's return has linear predictive power for today's return. If this
coefficient is close to zero, then yesterday's QQQ return provides little information about today's return. The R-squared value
measures how much variation in daily returns is explained by the model. 

Below ar1_prediction is the prediction on the test set from yesterday's returns. mean_prediction is a simple baseline prediction using the average training return.

In [8]:
y_test = test["return_t"]
X_test = test[["return_lag_1"]]

X_test = sm.add_constant(X_test, has_constant="add")

test["ar1_prediction"] = ar1_model.predict(X_test)
test["mean_prediction"] = y_train.mean()

test.head()

,return_t,return_lag_1,ar1_prediction,mean_prediction
Date,,,,
2024-02-27,0.002402,-0.000527,0.000798,0.000666
2024-02-28,-0.005339,0.002402,0.000471,0.000666
2024-02-29,0.008533,-0.005339,0.001336,0.000666
2024-03-01,0.014945,0.008533,-0.000214,0.000666
2024-03-04,-0.003575,0.014945,-0.000930,0.000666


Plot actual test returns against AR(1) predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    test.index,
    test["return_t"],
    label="Actual Test Returns",
    alpha=0.7,
)

ax.plot(
    test.index,
    test["ar1_prediction"],
    label="AR(1) Predictions",
    alpha=0.9,
)

ax.plot(
    test.index,
    test["mean_prediction"],
    label="Mean Baseline Prediction",
    linestyle="--"
)

ax.set_title("QQQ test returns vs AR(1) predictions")
ax.set_xlabel("Date")
ax.set_ylabel("Daily log return")
ax.legend()

plt.show()